## nb_03_1_player_info_silver

Cleans and validates table in bronze schema and and writes the result to sliver schema
Every cleaning/validation rule lives in its own function (defined once,
below) and is then applied one step at a time in its own cell, so each
intermediate result can be inspected before moving to the next step.

### Imports

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, count, when, lit, min, max,
    substring, concat, to_timestamp, to_date,
)

StatementMeta(, 6feb58ee-703c-426b-80ef-97b159593903, 17, Finished, Available, Finished, False)

### Load common db functions

In [ ]:
%run nb_00_dbutils

StatementMeta(, 6feb58ee-703c-426b-80ef-97b159593903, 26, Finished, Available, Finished, True)

### Parameters

`RUN_PIPELINE` controls whether the "Run the pipeline" steps below actually execute.
It defaults to `True` for normal, standalone runs of this notebook.

When this notebook is loaded from another notebook via `%run` (e.g. from a test
notebook), pass `RUN_PIPELINE = False` as a run parameter so only the function/config
definitions are loaded:

```
%run <notebook name> { "RUN_PIPELINE": false }
```

In [2]:
# This cell is tagged "parameters" so Fabric/Synapse can override it when the
# notebook is invoked with %run nb_02_game_silver { "RUN_PIPELINE": false }
RUN_PIPELINE: bool = True

StatementMeta(, 6feb58ee-703c-426b-80ef-97b159593903, 27, Finished, Available, Finished, False)

### Config

In [3]:
BRONZE_TABLE = "bronze.player_info"
SILVER_TABLE = "silver.player_info"

# Only these columns make it into the silver table
required_cols: list[str] = [
    "player_id",
    "firstname",
    "lastname",
    "birthdate",
    "primaryPosition",
    "nationality",
]

fill_null_cols: dict[str, str] = {
    "nationality": "None"
}

# Natural key used to de-duplicate rows
PRIMARY_KEYS: list[str] = ["player_id"]
DEDUPE_KEYS: list[str] = ["player_id"]

# Valid player position values as tuples
VALID_PRIMARY_POSITION: tuple[str] = ('LW', 'D', 'C', 'RW', 'G')


StatementMeta(, 6feb58ee-703c-426b-80ef-97b159593903, 28, Finished, Available, Finished, False)

## Run the pipeline
Each step runs in its own cell so the result can be inspected before moving on.

In [9]:
if RUN_PIPELINE:
    df = load_table(spark, BRONZE_TABLE, columns=required_cols)

    df = remove_duplicates(df, columns=DEDUPE_KEYS)
    df = fill_null_columns(df, keys=fill_null_cols)
    df = validate_no_nulls(df, columns=required_cols)
    df = validate_column_values(df, column="primaryPosition", values=VALID_PRIMARY_POSITION)

    # Convert birthdate to date
    df = df.withColumn(
        "birthdate",
        to_date(col("birthdate"))
    )

    # Validate primary key
    df = validate_primary_keys(df, keys=PRIMARY_KEYS)

    write_table(df, SILVER_TABLE)
    print("🏁 Silver load complete.")

StatementMeta(, 6feb58ee-703c-426b-80ef-97b159593903, 29, Finished, Available, Finished, False)

✅ Loaded bronze.player_info: 3925 rows


🔁 Removed 0 duplicate row(s) based on ['player_id']


🔁 Filled 8 null row(s) in 'nationality' with 'None'


✅ Primary key check passed — ['player_id'] is unique across 3925 row(s)


✅ Wrote silver.player_info (3925 rows, 6 columns)
🏁 Silver load complete.
